# Notebook 2: The tool-using Agent

### Purpose

A model knows a lot of things, but it cannot perform tasks on its own. It cannot inspect a set of papers or search for current trends unless the application supplies that capability. Tools close that gap. A tool is how an agent reaches outside its own head.

By the end you will be able to **explain how an agent extends it's capabilities via tool use**, becuase you'lle connect multiple tools and etach the model make decisions which tool to call.

## Outcomes

- Web search tool 
- Document retrieval tool
- Structured output tool

# Prerequisites

- Finished Notebook 1
- Same setup: runs with `mock` mode with no key; switch `PROVIDER` when ready 

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ms-cc-org/AGENTIC-AI-Workshop/blob/main/notebooks/02_tool_using_agent.ipynb)

# STEP 1 - Setup (Same block as Notebook 1)

In [ ]:
#  SETUP cell. This is for a provider-agnostic LLM client.
#  Read this cell; every notebook uses this same adapter.
#  Switch providers by changing PROVIDER.
#  Nothingelse in the notebook changes. an agent is a pattern, not a vendor.

# In Google Colab this cell installs the packages. Locally, run once.
try:
    import google.colab
    !pip -q install anthropic openai
    import os
    repo_url = "https://github.com/ms-cc-org/AGENTIC-AI-Workshop.git"
    repo_dir = "AGENTIC-AI-Workshop"

    if not os.path.exists(repo_dir):
        !git clone -q $repo_url
    if os.path.isdir(repo_dir):
        %cd $repo_dir
    else:
        print("Clone has failed or Skipped")
except ImportError:
    pass

import os, json

PROVIDER = "openai"   # "anthropic" | "openai" | "mock"
                      # "mock" runs this notebook with NO API key, 

MODELS = {
    "anthropic": "claude-haiku-4-5-20251001",  # cheapest current Claude model
    "openai":    "gpt-5.4-nano", # cheapest current GPT model
}

# API keys
# In Colab: click the key icon (left sidebar) -> add ANTHROPIC_API_KEY or
# OPENAI_API_KEY, then uncomment the line below and your specific provider to load them.
#from google.colab import userdata
#os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
#os.environ["OPENAI_API_KEY"] = userdata.get("MY_API_KEY")

# Normalized shapes we use everywhere (so the agent code never mentions a vendor):
#   message: {"role":"user","content":str}
#            {"role":"assistant","content":str|None,"tool_calls":[{id,name,args}]}
#            {"role":"tool","tool_call_id":id,"name":name,"content":str}
#   tool:    {"name":str,"description":str,"parameters":<json-schema>}
#   reply:   {"text":str,"tool_calls":[{id,name,args}],"stop_reason":str}


_MOCK_SCRIPT = []
def mock_reset(script):
    global _MOCK_SCRIPT; _MOCK_SCRIPT = list(script)
def _mock_call(messages, tools):
    if _MOCK_SCRIPT:
        kind, payload = _MOCK_SCRIPT.pop(0)
        if kind == "tool":
            return {"text":"", "tool_calls":[{"id":"mock_"+payload["name"],
                    "name":payload["name"], "args":payload["args"]}], "stop_reason":"tool_use"}
        return {"text":payload, "tool_calls":[], "stop_reason":"end"}
    last = next((m for m in reversed(messages) if m["role"] in ("user","tool")), {"content":""})
    return {"text": f"[mock reply to] {str(last.get('content',''))[:80]}",
            "tool_calls":[], "stop_reason":"end"}


def _to_anthropic(messages):
    out=[]
    for m in messages:
        if m["role"]=="user":
            out.append({"role":"user","content":m["content"]})
        elif m["role"]=="assistant":
            blocks=[]
            if m.get("content"): blocks.append({"type":"text","text":m["content"]})
            for tc in m.get("tool_calls",[]):
                blocks.append({"type":"tool_use","id":tc["id"],"name":tc["name"],"input":tc["args"]})
            out.append({"role":"assistant","content":blocks})
        elif m["role"]=="tool":
            out.append({"role":"user","content":[{"type":"tool_result",
                        "tool_use_id":m["tool_call_id"],"content":str(m["content"])}]})
    return out

def _to_openai(messages, system):
    out=[]
    if system: out.append({"role":"system","content":system})
    for m in messages:
        if m["role"]=="user":
            out.append({"role":"user","content":m["content"]})
        elif m["role"]=="assistant":
            msg={"role":"assistant","content":m.get("content") or None}
            if m.get("tool_calls"):
                msg["tool_calls"]=[{"id":tc["id"],"type":"function","function":{
                    "name":tc["name"],"arguments":json.dumps(tc["args"])}} for tc in m["tool_calls"]]
            out.append(msg)
        elif m["role"]=="tool":
            out.append({"role":"tool","tool_call_id":m["tool_call_id"],"content":str(m["content"])})
    return out

def call_llm(messages, tools=None, system=None, max_tokens=1024, temperature=0):
    '''One function. Any provider. This is portable across whatever API key you have or your institution has.'''
    if PROVIDER=="mock":
        return _mock_call(messages, tools)
    if PROVIDER=="anthropic":
        from anthropic import Anthropic
        client=Anthropic()
        kw=dict(model=MODELS["anthropic"], max_tokens=max_tokens,
                temperature=temperature, messages=_to_anthropic(messages))
        if system: kw["system"]=system
        if tools: kw["tools"]=[{"name":t["name"],"description":t["description"],
                                "input_schema":t["parameters"]} for t in tools]
        r=client.messages.create(**kw)
        text=""; calls=[]
        for b in r.content:
            if b.type=="text": text+=b.text
            elif b.type=="tool_use": calls.append({"id":b.id,"name":b.name,"args":b.input})
        return {"text":text,"tool_calls":calls,"stop_reason":r.stop_reason}
    if PROVIDER=="openai":
        from openai import OpenAI
        client=OpenAI()
        kw=dict(model=MODELS["openai"], messages=_to_openai(messages, system),
                temperature=temperature, max_completion_tokens=max_tokens)
        if tools: kw["tools"]=[{"type":"function","function":{"name":t["name"],
                    "description":t["description"],"parameters":t["parameters"]}} for t in tools]
        r=client.chat.completions.create(**kw)
        msg=r.choices[0].message
        calls=[{"id":tc.id,"name":tc.function.name,
                "args":json.loads(tc.function.arguments or "{}")} for tc in (msg.tool_calls or [])]
        return {"text":msg.content or "", "tool_calls":calls,
                "stop_reason":"tool_use" if calls else "end"}
    raise ValueError("Unknown PROVIDER: "+PROVIDER)

print(f"Setup ready. PROVIDER={PROVIDER!r}.")

In [ ]:
def run_agent(user_task, tools_spec, tool_fns, system=None, max_steps=6, verbose=True):
    '''The agent loop. The idea is:
       plan -> act (maybe call a tool) -> observe (feed the result back) -> repeat,
       until the model returns a final answer instead of a tool call.'''
    messages = [{"role":"user","content":user_task}]
    for step in range(1, max_steps+1):
        reply = call_llm(messages, tools=tools_spec, system=system)
        if reply["tool_calls"]:                                  # the model wants a tool
            messages.append({"role":"assistant","content":reply["text"],
                             "tool_calls":reply["tool_calls"]})
            for tc in reply["tool_calls"]:
                if verbose: print(f"  step {step}: tool `{tc['name']}` <- {tc['args']}")
                try:    result = tool_fns[tc["name"]](**tc["args"])   # run the real function
                except Exception as e: result = f"ERROR: {e}"
                messages.append({"role":"tool","tool_call_id":tc["id"],
                                 "name":tc["name"],"content":result})   # observe
        else:                                                    # no tool -> we are done
            if verbose: print(f"  step {step}: final answer")
            return reply["text"], messages
    return "Stopped: hit max_steps.", messages

# What is a tool?

Two parts, no technical jargon:
1. A **python function** that does the real, actual work.
2. **A Schema**(name, description and arguments) that tells the model the tool exists. The model reass the description to decide *when* to call it.

# Step 2 - Tool 1: Web search

Here we are exploring 2 ways:
- Default is DDGS (the `ddgs` library): It requires no signup, api required.
- **Tavily** API key 

In [ ]:
def search_web(query, k=3):
    # search the live web and gives short snippets. If tavily key available
    # it will be used, else DuckDuckGo (This is keyless)

    if os.environ.get("TAVILY_API_KEY"):
        try:
            from tavily import TavilyClient
            res = TavilyClient().search(query, max_results=k)
            return "\n".join(f"- {r['title']}: {r['content'][:250]}" for r in res["results"])
        except Exception as e:
            return f"(Tavily error: {e})"
        
    try:
        from ddgs import DDGS
        with DDGS() as d:
            hits = list(d.text(query, max_results=k))
        if hits:
            return "\n".join(f"- {h['title']}: {h['body'][:250]}" for h in hits)
    except Exception as e:
        return f"(web search unavailable offline: {e})"
    return "(no results)"

# Quick smoke test (works only with a network; fine if it prints the offline note)
print(search_web("OWASP guidelines"))

## Step 3 - Tool 2: document retrieval

This tool searches the files in `datasets/`. It's the agent's way to read *your* documents, the ones the model was never trained on.

**How it actually works (if your datasets need changing):**

`retrieve_docs(query, k)` is content-agnostic, it works for every file. It lowercases the query into a set
of words, then for every document counts how many of those words appear in it (an overlap score), and returns the top `k` documents. 
That is the whole method.

Two consequences for your datasets:

- Any `.md` or `.txt` file you drop in `datasets/` is picked up automatically. You do not need to reformat them. `readme.md` is skipped on purpose.
- It matches *words*, not *meaning*. A query for "grant due date" won't rank a doc that says "proposal deadline" unless the words overlap. That is the known limit of keyword search.

In [ ]:
import glob, os, re
from pathlib import Path

def get_datasets():
    p = Path.cwd()
    while p!= p.parent:
        if (p / "datasets").is_dir():
            return p/ "datasets"
        p = p.parent
    return Path("../datasets")

data_dir = get_datasets()

def load_corpus(folder=data_dir):
    folder = str(folder)
    docs = {}
    for path in glob.glob(os.path.join(folder, "*.md")) + glob.glob(os.path.join(folder, "*.txt")):
        name = os.path.basename(path)
        if name.lower() == "readme.md":
            continue
        with open(path) as f:
            docs[name] = f.read()
    if not docs:
        print(f"WARNING: no docs found in {folder!r}. If you are in colab, set repo_url right")
        docs = {"inline_note.md": "Synthetic note: log every query for a reproducible literature triage; measure retrieval precision and recall."}
    return docs

CORPUS = load_corpus()
print("Loaded", len(CORPUS), "documents from", str(data_dir), ":", list(CORPUS))

def retrieve_docs(query, k=2):
    #Return the k local documents whose text best matches the query and keyword score
    q = set(re.findall(r"[a-z]+", query.lower()))
    scored = []
    for name, text in CORPUS.items():
        words = re.findall(r"[a-z]+", text.lower())
        overlap = sum(1 for w in words if w in q)
        scored.append((overlap, name, text))
    scored.sort(reverse=True)
    top = [s for s in scored if s[0] > 0][:k]
    if not top:
        return "(no local document matched)"
    return "\n\n".join(f"[{name}] (score {score})\n{text[:280]}..." for score, name, text in top)

print("\nSample retrieval for 'AI funding deadline':\n")
print(retrieve_docs("AI funding deadline proposal"))

## Step 4 - Tool 3: structured output

Free text is hard for code to use. Often you want the agent to return the same
fields every time, in a fixed shape you can drop into a spreadsheet or database.
That is structured output: the model fills a **schema** instead of writing prose.

**How it works, and what it means for your datasets:**

Retrieval is content-agnostic, but structured output is **document-specific**. The
schema below (`title`, `deadline`, `award_max`) only makes sense for a *funding
call* like `NSF_AI_Funding_2026.md`. It would be meaningless on the IRB policy or
the RAG note, which have no "award amount."

So the rule is one schema per document *type*. You don't change your datasets for
this; you choose (or write) a schema that matches the fields the document actually
has. Below we validate a grant record in two layers: is it the right *shape*, and
are the values actually *usable* (a real date, a positive amount)?

In [ ]:
from datetime import date

# The shape the model should fill when it extracts a funding call.
extract_schema = {
    "type": "object",
    "properties": {
        "title":     {"type": "string"},
        "deadline":  {"type": "string", "description": "ISO date YYYY-MM-DD"},
        "award_max": {"type": "number", "description": "maximum award in dollars"},
    },
    "required": ["title", "deadline", "award_max"],
}

def _parse_money(value):
    #Turn '$600,000' or 600000 into a float. Returns None if it can't.
    try:
        return float(re.sub(r"[^0-9.]", "", str(value)))
    except ValueError:
        return None

def validate_record(record):
    """Two layers, not just 'are the keys present':
       1) structural: the required fields exist
       2) semantic:   the values are actually usable
    Returns (ok, problems)."""
    problems = []
    # layer 1 - structure
    for field in extract_schema["required"]:
        if field not in record:
            problems.append(f"missing field: {field}")
    # layer 2 - meaning (check what is present)
    if not str(record.get("title", "")).strip():
        problems.append("title is empty")
    d = str(record.get("deadline", ""))
    try:
        date.fromisoformat(d)
    except ValueError:
        problems.append(f"deadline is not a valid ISO date (YYYY-MM-DD): {d!r}")
    amount = _parse_money(record.get("award_max"))
    if amount is None or amount <= 0:
        problems.append(f"award_max is not a positive number: {record.get('award_max')!r}")
    return (len(problems) == 0, problems)

def record_grant(title, deadline, award_max, source=""):
    """TOOL: the agent calls this tool to save a structured grant record. 
    The typed arguments ARE the structured output; 
    the function validates before accepting, and returns a pass/fail 
    so the agent can fix a bad extraction."""
    record = {"title": title, "deadline": deadline, "award_max": award_max, "source": source}
    ok, problems = validate_record(record)
    return {"accepted": ok, "problems": problems, "record": record}

# A good record passes; a bad one is caught with reasons (not just "missing key").
good = record_grant("NSF AI in Research Infrastructure", "2026-11-03", "$600,000", "NSF_AI_Funding_2026.md")
bad  = record_grant("", "Nov 3 2026", "lots of money")
print("GOOD ->", good["accepted"], good["problems"])
print("BAD  ->", bad["accepted"], bad["problems"])

## Step 5 - the agent choice

Now give all three tools and give the agent a task that needs more than one tool. Then watch how the model reads the descriptions and picks a tool per step.

In [ ]:
tools_spec = [
    {"name":"search_web",
     "description":"Search the live web for current information and return snippets. Use for anything time-sensitive or not in local documents.",
     "parameters":{"type":"object","properties":{
        "query":{"type":"string"}, "k":{"type":"integer","description":"how many results"}},
        "required":["query"]}},
    {"name":"retrieve_docs",
     "description":"Search the local (institutional or system) documents (funding notes, policies, method notes). Use for internal/administrative questions.",
     "parameters":{"type":"object","properties":{
        "query":{"type":"string"}, "k":{"type":"integer"}}, "required":["query"]}},
    {"name":"record_grant",
     "description":"Save the key details of a funding call as a structured, validated record. "
                   "Call this AFTER reading the funding document, to turn prose into clean fields. "
                   "It checks that the date and amount are real and rejects a bad extraction.",
     "parameters":{"type":"object","properties":{
        "title":{"type":"string"},
        "deadline":{"type":"string","description":"ISO date YYYY-MM-DD"},
        "award_max":{"type":"number"},
        "source":{"type":"string","description":"the document name"}},
        "required":["title","deadline","award_max"]}},
]
tool_fns = {"search_web": search_web, "retrieve_docs": retrieve_docs, "record_grant": record_grant}

# The agent now does two steps: read the doc, THEN structure what it found.
# (In mock mode we define the steps. With a real PROVIDER the model decides.)
mock_reset([
    ("tool", {"name":"retrieve_docs", "args":{"query":"NSF AI funding deadline award", "k":1}}),
    ("tool", {"name":"record_grant", "args":{
        "title":"NSF AI in Research Infrastructure (NSF-26-XXXX)",
        "deadline":"2026-11-03", "award_max":"$600,000",
        "source":"NSF_AI_Funding_2026.md"}}),
    ("final", "Saved a validated grant record: NSF AI in Research Infrastructure, "
              "deadline 2026-11-03, award up to $600,000. The record passed validation."),
])

print("AGENT choosing among THREE tools:")
answer, transcript = run_agent(
    "Read our local NSF funding note, then save the program's title, deadline, and maximum award as a structured record.",
    tools_spec, tool_fns)
print("\nAnswer:\n", answer)
print("\nTool calls the agent made, in order:")
for m in transcript:
    if m["role"] == "assistant" and m.get("tool_calls"):
        for tc in m["tool_calls"]:
            print("  ->", tc["name"])


# Exercise - build a tool, then build your own

Tools are how you make the agent useful for *your* work. We'll build one together,
end to end, then you build a second one yourself.

## Part 1 (together): a length / limit checker

A real example from the given dataset is that the NSF call limits the Reproducibility
Statement to **2 pages**. Checking a section against its limit is boring and
error-prone, which makes it a perfect tool. A tool counts the same way every time.

The four steps to *any* tool (the same four, every time):
1. Write the **function** that does the work.
2. Write the **schema** that tells the model when and how to call it.
3. **Register** it in `tools_spec` and `tool_fns`.
4. **Ask** a question that needs it, and watch the agent call it.

Run the next cell to see all four.

## Part 2 (your turn): a tool researchers actually use

Pick one that maps to your work and build it with the same four steps. Ones that
come up again and again:

- **Requirements checker:** does a proposal mention every required section (Technical requirements,
  Reproducibility Statement, Future works)? Return what's missing.
- **Deadline countdown:** days between today and a deadline you pass in.
- **Budget check:** does a requested amount stay under the award cap?
- **Reading-level or jargon check** on an abstract, Introduction etc.

In [ ]:
# This is for PART 1 (example): a length checker
def check_length(text, max_words):
    #Count the words in a section and says in the limit or not. Checks only length, not quality.
    n = len(re.findall(r"\S+", text))
    return f"{n} words: {'WITHIN' if n <= max_words else 'OVER'} the {max_words}-word limit."

# schema (what the model reads to decide to call the function)
check_length_spec = {
    "name": "check_length",
    "description": "Count the words in a piece of text and check it against a limit. "
                   "Use for abstracts, statements, or any length-limited section.",
    "parameters": {"type": "object", "properties": {
        "text": {"type": "string"},
        "max_words": {"type": "integer"}}, "required": ["text", "max_words"]},
}

# register the tool
tools_spec.append(check_length_spec)
tool_fns["check_length"] = check_length

# Ask something that needs the tool 
draft = "We ensure reproducibility through deterministic fallbacks and full prompt logging. " * 20 #Input to check
mock_reset([
    ("tool", {"name":"check_length", "args":{"text":draft, "max_words":50}}),
    ("final", "Your reproducibility statement is over the 50-word limit; see the count above."),
])
ans, tr = run_agent("Is this reproducibility statement within a 50-word limit? " + draft,
                    tools_spec, tool_fns)
print("PART 1 ->", ans)

# Now you build a tool
# You can build this or anything of your choice.
# target: a requirements checker for a proposal. Fill in the TODOs.
def check_requirements(text, required_sections):
    """TODO: return which required sections are missing from the proposal text."""
    missing = []  # TODO: for each name in required_sections, test `name.lower() not in text.lower()`
    return {"missing": missing, "complete": len(missing) == 0}

# TODO 1: write the schema (name, description, parameters) like check_length_spec above
# TODO 2: tools_spec.append(your_spec);  tool_fns["check_requirements"] = check_requirements
# TODO 3: set a real PROVIDER and let the model choose or you need to write the mock_reset[...]
# TODO 4: run_agent("Does this proposal include a purpose, RQs and a Reproducibility Statement?", ...)
print("\nComplete the TODOs to add your own tool.")

## Reflection
- Which of your daily tasks are really "the model needs to *look something up* or *do* something"? Those can be tools.
- A tool's description is a tiny piece of writing that changes how the agent reacts. Where else do you need precise wording in your work?